## Part 1: Vector Database Fundamentals

In this notebook we will:
1. Connect to a local CockroachDB instance
2. Load an e-commerce product dataset from HuggingFace
3. Embed products with OpenAI and load them into CockroachDB
4. Run semantic searches, filtered searches, and RAG

**Key concepts**: embeddings, vector dimensions, cosine similarity, SQL filtering, RAG.

## Setup and Dependencies

In [ ]:
# If you ran pip install for requirements, do not uncomment this cell
#!pip install psycopg2-binary openai python-dotenv datasets

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL", "postgresql://root@localhost:26257/workshop?sslmode=disable")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("No OpenAI API key found. Update your .env file.")
else:
    print(f"Database URL: {DATABASE_URL}")
    print(f"OpenAI API Key: {OPENAI_API_KEY[:10]}...")

No OpenAI API key found. Update your .env file.


## Connect to CockroachDB

We are connecting to a local CockroachDB instance running via Docker Compose on port 26257. CockroachDB supports the pgvector-compatible `VECTOR` data type and cosine distance operator (`<=>`).

In [ ]:
import psycopg2

# Connecting to our local instance of our vector database
conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = True

# Create the workshop database if it doesn't exist, then reconnect to it
with conn.cursor() as cur:
    cur.execute("SELECT 1 FROM information_schema.schemata WHERE catalog_name = 'workshop'")
    if not cur.fetchone():
        cur.execute("CREATE DATABASE workshop")
conn.close()

# Reconnect targeting the workshop database
conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = True

# Check what tables exist
with conn.cursor() as cur:
    cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'")
    tables = [row[0] for row in cur.fetchall()]
print(f"Connected to CockroachDB. Existing tables: {tables}")

## Embedding Helper

CockroachDB stores vectors but does not generate embeddings. We will use OpenAI's embedding API ourselves, vectorize our objects and then pass them to the database.

In [ ]:
import openai

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Creating a functionb to vectorize our objects in batch
def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed a batch of texts using OpenAI."""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]

# Creating a function to vectorize a single object for testing
def embed_text(text: str) -> list[float]:
    """Embed a single text string using OpenAI."""
    return embed_texts([text])[0]

# Quick test
test_vec = embed_text("hello world")
print(f"Embedding dimension: {len(test_vec)}")

## Load the Dataset

We will use Qdrant's [H&M e-commerce products dataset](https://huggingface.co/datasets/Qdrant/hm_ecommerce_products) from HuggingFace. It has 105K products with rich metadata including product names, descriptions, colors, departments, categories, and more. 

We will take a 2,500 product sample and embed it ourselves into a new collection in memory.

In [ ]:
from datasets import load_dataset

# Pull in the data set from Hugging Face
full_dataset = load_dataset("Qdrant/hm_ecommerce_products", split="train")

# Showing data set info
print(f"Full dataset size: {len(full_dataset)}")
print(f"Columns: {full_dataset.column_names}")

In [ ]:
# Let's look at specific dta for one product
sample = full_dataset[0]
print(f"Product: {sample['prod_name']}")
print(f"Type: {sample['product_type_name']}")
print(f"Group: {sample['product_group_name']}")
print(f"Color: {sample['colour_group_name']}")
print(f"Department: {sample['department_name']}")
print(f"Section: {sample['section_name']}")
print(f"Index: {sample['index_name']}")
print(f"Description: {sample['detail_desc']}")
print(f"\nText used for embedding:\n{sample['text_to_embed']}")

In [ ]:
# Take 2,500 products
dataset = full_dataset.select(range(2500))
print(f"Working with {len(dataset)} products")

## Embed the Products

We embed the `text_to_embed` field, a pre-built string that combines product name, type, color, department, and description into a single passage. We batch the API calls for speed.

In [ ]:
# A nice tool for us to watch our embedding progress
from tqdm import tqdm

# Setting up our batches to embed
BATCH_SIZE = 100
all_texts = [item["text_to_embed"] for item in dataset]
all_vectors = []

# Pushing our objects to OpenAI for embedding in batches
for i in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="Embedding with OpenAI"):
    batch = all_texts[i:i + BATCH_SIZE]
    vectors = embed_texts(batch)
    all_vectors.extend(vectors)

print(f"Embedded {len(all_vectors)} products, dimension: {len(all_vectors[0])}")

## Create the Table and Import

In CockroachDB, we store products in a regular SQL table with a `VECTOR(1536)` column for embeddings. Metadata fields become regular columns that we can filter with standard SQL `WHERE` clauses.

In [ ]:
# A BAD idea to run this in production
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS products")

    cur.execute("""
        CREATE TABLE products (
            id INT PRIMARY KEY,
            prod_name STRING,
            product_type_name STRING,
            product_group_name STRING,
            colour_group_name STRING,
            department_name STRING,
            section_name STRING,
            index_name STRING,
            detail_desc STRING,
            garment_group_name STRING,
            embedding VECTOR(1536)
        )
    """)

print("Table 'products' created (1536-dim VECTOR column)")

In [ ]:
# Insert products one at a time (CockroachDB recommends avoiding large vector batches)
with conn.cursor() as cur:
    for i, item in enumerate(tqdm(dataset, desc="Inserting products")):
        cur.execute("""
            INSERT INTO products (id, prod_name, product_type_name, product_group_name,
                colour_group_name, department_name, section_name, index_name,
                detail_desc, garment_group_name, embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (i, item["prod_name"], item["product_type_name"], item["product_group_name"],
              item["colour_group_name"], item["department_name"], item["section_name"],
              item["index_name"], item["detail_desc"], item["garment_group_name"],
              str(all_vectors[i])))

# Sanity check
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM products")
    count = cur.fetchone()[0]
print(f"\nImported {count} products")

# Create vector index for cosine similarity search
with conn.cursor() as cur:
    cur.execute("SET CLUSTER SETTING feature.vector_index.enabled = true")
    cur.execute("SET sql_safe_updates = false")
    cur.execute("CREATE VECTOR INDEX ON products (embedding vector_cosine_ops)")
print("Vector index created (cosine distance)")

## Semantic Search

Vector search finds results based on meaning. We embed the query with the same OpenAI model and find the closest products by cosine similarity.

In [ ]:
# Embedding our query with the same model we used to embed our data
query_vector = embed_text("dark winter jacket")

# Sending our embedded query to the database
with conn.cursor() as cur:
    cur.execute("""
        SELECT prod_name, product_type_name, colour_group_name,
               detail_desc, embedding <=> %s AS distance
        FROM products
        ORDER BY embedding <=> %s
        LIMIT 5
    """, (str(query_vector), str(query_vector)))
    results = cur.fetchall()

# Display our top 5 results
for i, row in enumerate(results, 1):
    print(f"{i}. {row[0]}")
    print(f"   Type: {row[1]} | Color: {row[2]}")
    print(f"   Distance: {row[4]:.4f}")
    print(f"   {row[3][:100]}...")
    print()

## Filtered Search

SQL WHERE clauses constrain the search space before vector similarity runs. This is critical for production — you almost always want to narrow your results and have the ability to utilize your metadata columns to find the nuance not caught by semantic search.

In [ ]:
# Vectorize our query
query_vector = embed_text("casual summer top")

# Add our column filter
with conn.cursor() as cur:
    cur.execute("""
        SELECT prod_name, colour_group_name, product_type_name,
               embedding <=> %s AS distance
        FROM products
        WHERE colour_group_name = %s
        ORDER BY embedding <=> %s
        LIMIT 5
    """, (str(query_vector), "White", str(query_vector)))
    results = cur.fetchall()

for i, row in enumerate(results, 1):
    print(f"{i}. {row[0]} ({row[1]})")
    print(f"   Type: {row[2]} | Distance: {row[3]:.4f}")
    print()

### Combining Multiple Filters

Combine filters with `must` (AND), `should` (OR), and `must_not` (NOT).

In [ ]:
query_vector = embed_text("comfortable basics")

with conn.cursor() as cur:
    cur.execute("""
        SELECT prod_name, section_name, colour_group_name,
               embedding <=> %s AS distance
        FROM products
        WHERE index_name = %s AND colour_group_name != %s
        ORDER BY embedding <=> %s
        LIMIT 5
    """, (str(query_vector), "Ladieswear", "Black", str(query_vector)))
    results = cur.fetchall()

for i, row in enumerate(results, 1):
    print(f"{i}. {row[0]}")
    print(f"   {row[1]} | {row[2]} | Distance: {row[3]:.4f}")
    print()

## RAG: Retrieve and Generate

CockroachDB handles retrieval. For generation, we pass the retrieved context to an LLM. This is basic RAG at its core.

In [ ]:
# Create our query
query = "I need a formal outfit for a business meeting"

# Vectorize it
query_vector = embed_text(query)

# Pull similar products from our database
with conn.cursor() as cur:
    cur.execute("""
        SELECT prod_name, product_type_name, colour_group_name, detail_desc
        FROM products
        ORDER BY embedding <=> %s
        LIMIT 5
    """, (str(query_vector),))
    results = cur.fetchall()

# Build context from retrieved products
context = "\n".join(
    f"- {row[0]} ({row[1]}, {row[2]}): {row[3]}"
    for row in results
)

# Send our original query with our query results and a system prompt to our generative model
response = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a personal shopping assistant. Recommend products based on the available inventory."},
        {"role": "user", "content": f"Customer request: {query}\n\nAvailable products:\n{context}\n\nRecommend the best options and explain why."},
    ],
)

print("=== Shopping Assistant ===")
print(response.choices[0].message.content)

## Summary

What we learned:

- **Embedding is the translation step** — turning text into vectors that capture meaning
- **Vector search** finds semantically similar results by comparing these vectors
- **Payload filters** constrain the search space with metadata (`must`, `should`, `must_not`)
- **RAG** = retrieve context from the vector database, then generate answers with an LLM
- **Your query embedding model must match your document embedding model** — you can't mix and match

Next up: we will build agents that automate the hard parts,  dynamic filtering and query decomposition.

In [ ]:
conn.close()